# Hub-and-Spoke Multi-Agent Architecture on AWS Bedrock
**Region:** `ap-southeast-1` | **Model:** Claude Sonnet 4.6 via Bedrock Converse API

A fully decoupled research pipeline where each agent has a single, bounded responsibility.

In [12]:
!pip install -q boto3 pydantic rich

In [ ]:
import json
import time
import boto3
from pydantic import BaseModel, Field, ValidationError
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.text import Text
from botocore.exceptions import ClientError

console = Console()

bedrock_client = boto3.client("bedrock-runtime", region_name="ap-southeast-1")
MODEL_ID = "global.anthropic.claude-sonnet-4-6"

console.print(Panel(
    f"[green]✓[/green] Bedrock client ready\n"
    f"[cyan]Region:[/cyan] ap-southeast-1\n"
    f"[cyan]Model :[/cyan] {MODEL_ID}",
    title="Setup"
))

╭───────────────────────────────────────────────────── Setup ─────────────────────────────────────────────────────╮
│ ✓ Bedrock client ready                                                                                          │
│ Region: ap-southeast-1                                                                                          │
│ Model : apac.anthropic.claude-sonnet-4-6                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Architecture

```
                        ┌─────────────────────────────┐
                        │         HUB (Coordinator)    │
                        │  Deconstructs goal → subtasks│
                        └──────────────┬──────────────┘
                                       │
               ┌───────────────────────┼───────────────────────┐
               ▼                                               ▼
   ┌───────────────────────┐                   ┌───────────────────────┐
   │  SPOKE 1              │                   │  SPOKE 2              │
   │  Web Researcher       │                   │  Document Analyzer    │
   │  [simulate_web_search]│                   │  [extract_doc_data]   │
   └──────────┬────────────┘                   └────────────┬──────────┘
              │                                             │
              └─────────────────┬───────────────────────────┘
                                ▼
                   ┌────────────────────────┐
                   │  SPOKE 3 — Synthesizer  │
                   │  RAW DATA → FACT BASE   │
                   │  (JSON only, no prose)  │
                   └────────────┬───────────┘
                                │  ◄── hard wall: schema-validated JSON
                                ▼
                   ┌────────────────────────┐
                   │  SPOKE 4 — Report Writer│
                   │  FACT BASE → MARKDOWN   │
                   │  (prose only, no infer) │
                   └────────────────────────┘
```

**Data flow:** `Goal` → `Hub` → `Subtasks` → `Spokes 1+2` → `ResearchState.raw_*` → `Synthesizer` → `ResearchState.fact_base` → `ReportWriter` → `ResearchState.final_report`

In [14]:
# --- Pydantic models for strict state management ---

class Fact(BaseModel):
    claim: str
    source: str
    confidence: float = Field(ge=0.0, le=1.0)

class Conflict(BaseModel):
    claims: list[str]
    sources: list[str]
    resolution: str | None = None

class FactBase(BaseModel):
    topic: str
    facts: list[Fact]
    conflicts: list[Conflict]
    confidence_score: float = Field(ge=0.0, le=1.0)

class ResearchState(BaseModel):
    goal: str
    raw_web: list[str] = []
    raw_doc: list[str] = []
    fact_base: dict | None = None
    final_report: str | None = None

console.print("[green]✓[/green] ResearchState and FactBase models defined")

✓ ResearchState and FactBase models defined

In [15]:
# --- Simulated tool implementations (realistic conflicting SE Asia EV data) ---

WEB_DATA = {
    "default": (
        "[Reuters, Jan 2026] BYD leads Southeast Asian EV market with 38% share. "
        "[Nikkei, Feb 2026] BYD's SEA share revised to 31% after audit. "
        "[Bloomberg, Mar 2026] Wuling holds 22% SEA EV share; strong in Vietnam/Indonesia. "
        "[TechCrunch, Mar 2026] Local brand VinFast claims 18% SEA market share citing internal data. "
        "[FT, Apr 2026] Analysts dispute VinFast figure; independent estimate puts them at 11%. "
        "[S&P Global, Apr 2026] Toyota BEV pivot gains traction; 9% SEA share. "
        "[WSJ, May 2026] Total SEA EV sales grew 67% YoY to 1.2M units in 2025."
    )
}

DOC_DATA = {
    "default": (
        "[ASEAN Automotive Report Q1 2026]\n"
        "- BYD: 34.5% market share (blended Q1 average across 6 SEA nations)\n"
        "- Wuling/SGMW: 21.8%\n"
        "- VinFast: 12.3% (Vietnam-heavy; 4.1% ex-Vietnam)\n"
        "- Toyota (BEV): 8.7%\n"
        "- Others (Honda, Hyundai, Chery, local): 22.7%\n"
        "Note: Methodology differs from self-reported figures. Reuters/Nikkei discrepancy for BYD "
        "likely due to Reuters including fleet/B2B orders not in retail registrations."
    )
}

def simulate_web_search(query: str) -> str:
    return WEB_DATA.get(query.lower(), WEB_DATA["default"])

def extract_document_data(topic: str) -> str:
    return DOC_DATA.get(topic.lower(), DOC_DATA["default"])

TOOL_HANDLERS = {
    "simulate_web_search": simulate_web_search,
    "extract_document_data": extract_document_data,
}

# --- Bedrock toolConfig schema ---
TOOL_CONFIG = {
    "tools": [
        {
            "toolSpec": {
                "name": "simulate_web_search",
                "description": "Search the web for news, reports, and data on a topic.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "query": {"type": "string", "description": "Search query"}
                        },
                        "required": ["query"],
                    }
                },
            }
        },
        {
            "toolSpec": {
                "name": "extract_document_data",
                "description": "Extract structured data from internal documents or reports on a topic.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "topic": {"type": "string", "description": "Topic to extract data about"}
                        },
                        "required": ["topic"],
                    }
                },
            }
        },
    ]
}

console.print("[green]✓[/green] Tool definitions and handlers ready")

✓ Tool definitions and handlers ready

In [16]:
HUB_PROMPT = """
You are a research coordinator. Your ONLY job is to decompose a research goal into subtasks.
Return ONLY a JSON array. Each element must have exactly two keys: "agent" and "task".
Valid agent values: "WebResearcher", "DocAnalyzer".
Do not include any prose, explanation, or markdown fencing. Raw JSON only.

Example output:
[
  {"agent": "WebResearcher", "task": "Find 2026 EV market share data for Southeast Asia"},
  {"agent": "DocAnalyzer", "task": "Extract EV market share from internal ASEAN reports"}
]
"""

WEB_RESEARCHER_PROMPT = """
You are a web research agent. Use the simulate_web_search tool to retrieve data.
Return the raw text findings exactly as retrieved. Do not summarize, interpret, or filter.
Do not generate prose beyond what the tool returns.
"""

DOC_ANALYZER_PROMPT = """
You are a document analysis agent. Use the extract_document_data tool to pull structured data.
Return the raw extracted text exactly as retrieved. Do not summarize, interpret, or filter.
Do not generate prose beyond what the tool returns.
"""

SYNTHESIZER_PROMPT = """
You are a data synthesis agent. You receive raw research data from multiple sources.
Your ONLY job is to produce a valid JSON object matching this exact schema:

{
  "topic": "<string>",
  "facts": [
    {"claim": "<string>", "source": "<string>", "confidence": <0.0-1.0>}
  ],
  "conflicts": [
    {"claims": ["<string>", ...], "sources": ["<string>", ...], "resolution": "<string or null>"}
  ],
  "confidence_score": <0.0-1.0>
}

STRICT CONSTRAINTS:
- Output ONLY valid JSON. No prose. No markdown fencing. No explanation.
- Flag ALL numeric contradictions between sources as conflicts.
- Assign lower confidence to self-reported figures with no independent corroboration.
- Do not infer, speculate, or generate facts not present in the input data.
"""

REPORT_WRITER_PROMPT = """
You are an executive report writer. You receive a JSON Fact Base and write a polished Markdown report.

STRICT CONSTRAINTS:
- Use ONLY facts and conflicts present in the JSON. Do not infer, research, or add new information.
- Structure the report with: Executive Summary, Key Findings, Data Conflicts & Resolution, Confidence Assessment.
- Use tables for market share data where applicable.
- Be precise about source attribution. Call out unresolved conflicts explicitly.
- Do not begin with 'I' or describe your own actions.
"""

console.print("[green]✓[/green] System prompts defined for all 5 agents")

✓ System prompts defined for all 5 agents

In [17]:
def call_agent(
    system: str,
    messages: list[dict],
    tools: dict | None = None,
    max_retries: int = 5,
) -> str:
    """
    Calls the Bedrock Converse API with exponential backoff on throttling.
    Handles tool_use stop reason automatically by executing tools and looping.
    Returns the final assistant text.
    """
    messages = list(messages)  # immutable copy

    for attempt in range(max_retries):
        try:
            kwargs = {
                "modelId": MODEL_ID,
                "system": [{"text": system}],
                "messages": messages,
                "inferenceConfig": {"maxTokens": 4096},
            }
            if tools:
                kwargs["toolConfig"] = tools

            response = bedrock_client.converse(**kwargs)
            stop_reason = response["stopReason"]
            output_message = response["output"]["message"]

            if stop_reason == "end_turn":
                return output_message["content"][0]["text"]

            if stop_reason == "tool_use":
                # Execute all requested tools
                tool_results = []
                for block in output_message["content"]:
                    if block.get("toolUse"):
                        tool_use = block["toolUse"]
                        fn = TOOL_HANDLERS.get(tool_use["name"])
                        if not fn:
                            raise ValueError(f"Unknown tool: {tool_use['name']}")
                        result = fn(**tool_use["input"])
                        console.print(f"  [dim]→ tool:[/dim] [yellow]{tool_use['name']}[/yellow]({tool_use['input']})")
                        tool_results.append({
                            "toolResult": {
                                "toolUseId": tool_use["toolUseId"],
                                "content": [{"text": result}],
            				}
                        })

                # Append assistant turn + all tool results, then loop
                messages.append(output_message)
                messages.append({"role": "user", "content": tool_results})
                continue  # re-enter loop without incrementing attempt

            raise RuntimeError(f"Unexpected stop_reason: {stop_reason}")

        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("ThrottlingException", "ServiceUnavailableException", "ModelNotReadyException"):
                wait = 2 ** attempt
                console.print(f"  [red]⚠ {code}. Retrying in {wait}s (attempt {attempt+1}/{max_retries})...[/red]")
                time.sleep(wait)
            else:
                raise

    raise RuntimeError("Exceeded max retries due to Bedrock API errors.")

console.print("[green]✓[/green] call_agent() helper ready")

✓ call_agent() helper ready

In [18]:
def run_hub(goal: str) -> list[dict]:
    console.print(Panel(f"[bold cyan]HUB[/bold cyan] Decomposing goal:\n{goal}", border_style="cyan"))

    response = call_agent(
        system=HUB_PROMPT,
        messages=[{"role": "user", "content": [{"text": goal}]}],
    )

    # Strip markdown fences if model wraps JSON anyway
    cleaned = response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    subtasks = json.loads(cleaned)

    for i, t in enumerate(subtasks, 1):
        console.print(f"  [cyan]{i}.[/cyan] [{t['agent']}] {t['task']}")

    return subtasks

console.print("[green]✓[/green] run_hub() defined")

✓ run_hub() defined

In [19]:
def run_spokes(subtasks: list[dict], state: ResearchState) -> ResearchState:
    raw_web = list(state.raw_web)
    raw_doc = list(state.raw_doc)

    for subtask in subtasks:
        agent = subtask["agent"]
        task = subtask["task"]

        if agent == "WebResearcher":
            console.print(Panel(f"[bold green]SPOKE 1 — Web Researcher[/bold green]\n{task}", border_style="green"))
            result = call_agent(
                system=WEB_RESEARCHER_PROMPT,
                messages=[{"role": "user", "content": [{"text": task}]}],
                tools=TOOL_CONFIG,
            )
            raw_web.append(result)

        elif agent == "DocAnalyzer":
            console.print(Panel(f"[bold magenta]SPOKE 2 — Document Analyzer[/bold magenta]\n{task}", border_style="magenta"))
            result = call_agent(
                system=DOC_ANALYZER_PROMPT,
                messages=[{"role": "user", "content": [{"text": task}]}],
                tools=TOOL_CONFIG,
            )
            raw_doc.append(result)

        else:
            console.print(f"[yellow]⚠ Unknown agent type '{agent}', skipping.[/yellow]")

    return state.model_copy(update={"raw_web": raw_web, "raw_doc": raw_doc})

console.print("[green]✓[/green] run_spokes() defined")

✓ run_spokes() defined

In [20]:
def run_synthesizer(state: ResearchState) -> ResearchState:
    console.print(Panel("[bold yellow]SPOKE 3 — Synthesizer[/bold yellow]\nConsolidating raw data → Fact Base JSON", border_style="yellow"))

    combined = (
        "=== WEB RESEARCH ===\n" + "\n---\n".join(state.raw_web) +
        "\n\n=== DOCUMENT DATA ===\n" + "\n---\n".join(state.raw_doc)
    )

    response = call_agent(
        system=SYNTHESIZER_PROMPT,
        messages=[{"role": "user", "content": [{"text": combined}]}],
    )

    cleaned = response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    try:
        fact_base = FactBase.model_validate_json(cleaned)
        console.print(f"  [green]✓[/green] Fact Base validated: {len(fact_base.facts)} facts, {len(fact_base.conflicts)} conflicts")
    except ValidationError as e:
        console.print(f"  [red]✗ Pydantic validation failed:[/red]\n{e}")
        raise

    return state.model_copy(update={"fact_base": fact_base.model_dump()})

console.print("[green]✓[/green] run_synthesizer() defined")

✓ run_synthesizer() defined

In [21]:
def run_report_writer(state: ResearchState) -> ResearchState:
    console.print(Panel("[bold blue]SPOKE 4 — Report Writer[/bold blue]\nGenerating Markdown report from Fact Base", border_style="blue"))

    fact_base_json = json.dumps(state.fact_base, indent=2)

    report = call_agent(
        system=REPORT_WRITER_PROMPT,
        messages=[{"role": "user", "content": [{"text": fact_base_json}]}],
    )

    console.print("  [green]✓[/green] Report generated")
    return state.model_copy(update={"final_report": report})

console.print("[green]✓[/green] run_report_writer() defined")

✓ run_report_writer() defined

In [23]:
# ── End-to-End Execution ──────────────────────────────────────────────────────

GOAL = "Analyze conflicting 2026 market share data for Southeast Asian EV manufacturers"

console.rule("[bold]Pipeline Start[/bold]")

state = ResearchState(goal=GOAL)

# Stage 1 — Hub decomposes goal
subtasks = run_hub(GOAL)

# Stage 2 — Discovery spokes gather raw data
state = run_spokes(subtasks, state)

# Stage 3 — Synthesizer validates & consolidates into JSON Fact Base
state = run_synthesizer(state)

# Stage 4 — Report Writer produces final Markdown
state = run_report_writer(state)

console.rule("[bold green]Pipeline Complete[/bold green]")

───────────────────────────────────────────────── Pipeline Start ──────────────────────────────────────────────────

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ HUB Decomposing goal:                                                                                           │
│ Analyze conflicting 2026 market share data for Southeast Asian EV manufacturers                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ValidationException: An error occurred (ValidationException) when calling the Converse operation: The provided model identifier is invalid.

In [ ]:
# Inspect the intermediate Fact Base (the hard wall between Synthesizer and Report Writer)
console.print(Panel(
    json.dumps(state.fact_base, indent=2),
    title="[yellow]Fact Base JSON (Synthesizer output)[/yellow]",
    border_style="yellow",
))

In [ ]:
# Render the final executive report
console.rule("[bold blue]Final Report[/bold blue]")
console.print(Markdown(state.final_report))